In [1]:
config_file = "./02_other_inputs_preprocessing/02_Config_file/config_file.json"

# Data preprocessing

This notebook is expected to be executed to preprocess the Glassdoor dataset (enhanced with traditional and non-traditional NLP methods as well as location, company and stock data).

## 1. Import libraries and functions, and initialize code

In [ ]:
!pip install pandas numpy scikit-learn openpyxl

In [2]:
## Import libraries

# Pandas
import pandas as pd

# Numpy
import numpy as np

# Json
import json

# Operating system
import os

# Sci-kit learn
from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

from sklearn.preprocessing import OneHotEncoder

from sklearn.preprocessing import StandardScaler

# Warnings
import warnings

# Pickle
import pickle

# Datetime
from datetime import datetime


## Configure libraries

# Pandas rows
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)

# Warnings
warnings.filterwarnings('ignore')

In [3]:
# Dictionary to be saved as a pickle object
dict_out = {}

## 2. Read data

### 2.1. Actually read data

In [4]:
# Read the config file
with open(config_file, 'r') as file:
    config = json.load(file)

In [5]:
# Get review list
reviews_path = config['general']['input files']['reviews path']
files_reviews = os.listdir(reviews_path)

# Shorten list if necessary
if config['general']['execution type'] == "test":
    files_reviews = files_reviews[0:5]
elif config['general']['execution type'] == "real":
    files_reviews = files_reviews
else:
    raise Exception("Invalid execution type")

In [6]:
# Actually read files
reviews = pd.concat([pd.read_csv(reviews_path + "/" + file) for file in files_reviews], ignore_index=True)

### 2.2. Check data read

In [7]:
# Check size of reviews
print("Complete dataset size " + str(reviews.shape))

# Show head of dataframe
display(reviews.head(5))

Complete dataset size (356594, 283)


,Unnamed: 0,review_id,summary,date,job_title,overall_rating,pros,cons,author_location,wl_balance,culture_values,diversity_inclusion,career_opportunities,compensation_benefits,senior_management,company,summary_charlen,summary_wordlen,pros_charlen,pros_wordlen,cons_charlen,cons_wordlen,summary_ld_PDT,summary_ld_IN,summary_ld_``,summary_ld_JJS,summary_ld_RBS,summary_ld_NNP,summary_ld_DT,summary_ld_EX,summary_ld_NNS,summary_ld_CC,summary_ld_VBD,summary_ld_.,summary_ld_(,summary_ld_UH,summary_ld_VBN,summary_ld_RP,summary_ld_JJR,summary_ld_WDT,"summary_ld_,",summary_ld_VBZ,summary_ld_MD,summary_ld_PRP,summary_ld_PRP$,summary_ld_$,summary_ld_WP,summary_ld_VBG,summary_ld_NN,summary_ld_CD,summary_ld_),summary_ld_RBR,summary_ld_VBP,summary_ld_WRB,summary_ld_NNPS,summary_ld_TO,summary_ld_RB,summary_ld_VB,summary_ld_:,summary_ld_JJ,pros_ld_PDT,pros_ld_IN,pros_ld_``,pros_ld_JJS,pros_ld_RBS,pros_ld_NNP,pros_ld_DT,pros_ld_EX,pros_ld_NNS,pros_ld_CC,pros_ld_VBD,pros_ld_POS,pros_ld_.,pros_ld_(,pros_ld_UH,pros_ld_VBN,pros_ld_'',pros_ld_RP,pros_ld_JJR,pros_ld_WDT,pros_ld_FW,"pros_ld_,",pros_ld_VBZ,pros_ld_MD,pros_ld_PRP,pros_ld_PRP$,pros_ld_$,pros_ld_WP,pros_ld_VBG,pros_ld_NN,pros_ld_CD,pros_ld_),pros_ld_SYM,pros_ld_RBR,pros_ld_VBP,pros_ld_WRB,pros_ld_NNPS,pros_ld_TO,pros_ld_RB,pros_ld_VB,pros_ld_:,pros_ld_JJ,cons_ld_PDT,cons_ld_IN,cons_ld_``,cons_ld_JJS,cons_ld_RBS,cons_ld_DT,cons_ld_NNP,cons_ld_EX,cons_ld_NNS,cons_ld_CC,cons_ld_VBD,cons_ld_POS,cons_ld_.,cons_ld_(,cons_ld_'',cons_ld_VBN,cons_ld_UH,cons_ld_RP,cons_ld_JJR,cons_ld_WDT,cons_ld_FW,"cons_ld_,",cons_ld_VBZ,cons_ld_MD,cons_ld_PRP,cons_ld_PRP$,cons_ld_$,cons_ld_WP,cons_ld_VBG,cons_ld_NN,cons_ld_CD,cons_ld_),cons_ld_SYM,cons_ld_RBR,cons_ld_VBP,cons_ld_WRB,cons_ld_NNPS,cons_ld_TO,cons_ld_RB,cons_ld_VB,cons_ld_:,cons_ld_JJ,pros_emolex_fear,pros_emolex_anger,pros_emolex_anticip,pros_emolex_trust,pros_emolex_surprise,pros_emolex_positive,pros_emolex_negative,pros_emolex_sadness,pros_emolex_disgust,pros_emolex_joy,cons_emolex_fear,cons_emolex_anger,cons_emolex_anticip,cons_emolex_trust,cons_emolex_surprise,cons_emolex_positive,cons_emolex_negative,cons_emolex_sadness,cons_emolex_disgust,cons_emolex_joy,pros_nltk_sia,cons_nltk_sia,ODI_related_vocab_pros,ODI_related_vocab_cons,pros_empath_help,pros_empath_office,pros_empath_money,pros_empath_domestic_work,pros_empath_sleep,pros_empath_occupation,pros_empath_family,pros_empath_vacation,pros_empath_health,pros_empath_swearing_terms,pros_empath_leisure,pros_empath_suffering,pros_empath_wealthy,pros_empath_exercise,pros_empath_home,pros_empath_rage,pros_empath_fun,pros_empath_negative_emotion,pros_empath_payment,pros_empath_achievement,cons_empath_help,cons_empath_office,cons_empath_money,cons_empath_domestic_work,cons_empath_sleep,cons_empath_occupation,cons_empath_family,cons_empath_vacation,cons_empath_health,cons_empath_swearing_terms,cons_empath_leisure,cons_empath_suffering,cons_empath_wealthy,cons_empath_exercise,cons_empath_home,cons_empath_rage,cons_empath_fun,cons_empath_negative_emotion,cons_empath_payment,cons_empath_achievement,pros_sim_ODI_Depressed mood,cons_sim_ODI_Depressed mood,pros_sim_ODI_Sleep alterations,cons_sim_ODI_Sleep alterations,pros_sim_ODI_Fatigue,cons_sim_ODI_Fatigue,pros_sim_ODI_Worthlessness,cons_sim_ODI_Worthlessness,pros_sim_ODI_Anhedonia,cons_sim_ODI_Anhedonia,pros_sim_ODI_Appetite alterations,cons_sim_ODI_Appetite alterations,pros_sim_ODI_Cognitive impairment,cons_sim_ODI_Cognitive impairment,pros_sim_ODI_Psycomotor alterations,cons_sim_ODI_Psycomotor alterations,pros_sim_ODI_Suicidal ideation,cons_sim_ODI_Suicidal ideation,pros_sim_JDI_Work itself,cons_sim_JDI_Work itself,pros_sim_JDI_Pay,cons_sim_JDI_Pay,pros_sim_JDI_Promotion,cons_sim_JDI_Promotion,pros_sim_JDI_Supervision,cons_sim_JDI_Supervision,pros_sim_JDI_Coworkers,cons_sim_JDI_Coworkers,overall_rating_cat,Company,author_location_filledhq,author_location_filledhq_state,Acronym,State name,GDP per capita,Population,Precipitation,AverageTemperatureF,AverageT

In [8]:
# Check duplicates
if reviews['review_id'].duplicated().sum() == 0:
    "No duplicates found"
else:
    raise Exception("Duplicates in dataset")

## 3. Split dataset in training - validation - testing

### 3.1. Actually split

In [9]:
# Calculate train - validate - test proportion
train_prop = config['TVT split']['data_split_percentages']['train']
test_prop = config['TVT split']['data_split_percentages']['test']

# Print values to check
print("Train proportion: " + str(config['TVT split']['data_split_percentages']['train']))
print("Test proportion: " + str(config['TVT split']['data_split_percentages']['test']))

Train proportion: 0.8
Test proportion: 0.2


In [10]:
# Separate data
random_seed = config['general']['random seed']
reviews_train, reviews_test = train_test_split(reviews, test_size=test_prop, random_state=random_seed)

# Print proportions to check
print("Train proportion: " + str(len(reviews_train) / len(reviews)))
print("Test proportion: " + str(len(reviews_test) / len(reviews)))

Train proportion: 0.7999994391380674
Test proportion: 0.20000056086193263


### 3.2. Check consistency

In [11]:
def check_dataframes(dataframes, iqr_threshold, cat_max, exclude_columns, cats_ignore):
    
    # Filter columns based on the exclusion list
    columns_to_check = [col for col in dataframes[0].columns if col not in exclude_columns]

    # Check numerical columns
    for col in columns_to_check:
        
        # Skip non-numerical columns
        if not(dataframes[0][col].dtype in [np.float64, np.int64]):
            continue
        elif (dataframes[0][col]==0).sum() / len(dataframes[0]) > 0.5:
            continue
            
        # Calculate interquartile ranges
        iqr_values = [df[col].quantile(0.75) - df[col].quantile(0.25) for df in dataframes]
        max_iqr = max(iqr_values)
        min_iqr = min(iqr_values)

        iqr_ratio = abs((max_iqr - min_iqr) / max_iqr)

        if iqr_ratio > iqr_threshold:
            print(col)
            raise ValueError(f"Numerical column '{col}' does not satisfy conditions.")


    # Check categorical columns
    for col in columns_to_check:
        if dataframes[0][col].dtype == 'O':  # Check if the column is of object type (categorical)
            # Filter categories with frequency under 10%
            category_counts = dataframes[0][col].value_counts(normalize=True)
            valid_categories = category_counts[category_counts >= cats_ignore].index

            for category in valid_categories:
                min_count = min(df[col].value_counts().get(category, 0) / len(df) for df in dataframes)
                max_count = max(df[col].value_counts().get(category, 0) / len(df) for df in dataframes)

                cat_diff = abs((max_count - min_count) / max_count)

                if cat_diff > cat_max:
                    for df in dataframes:
                        print(df[col].value_counts().get(category, 0) / len(df))
                    raise ValueError(f"Categorical column '{col}', category '{category}' does not satisfy conditions.")

    # If all checks pass, print the success message
    print("All checks satisfactory.")

In [12]:
# Calculate value counts
vc_output_all = pd.DataFrame(reviews['overall_rating'].value_counts() / len(reviews)).sort_index(axis=0, ascending=True, inplace=False).rename(columns={'count': 'all'})
vc_output_train = pd.DataFrame(reviews_train['overall_rating'].value_counts() / len(reviews_train)).sort_index(axis=0, ascending=True, inplace=False).rename(columns={'count': 'train'})
vc_output_test = pd.DataFrame(reviews_test['overall_rating'].value_counts() / len(reviews_test)).sort_index(axis=0, ascending=True, inplace=False).rename(columns={'count': 'test'})

# Join all dataframes
vc_output = vc_output_all.merge(vc_output_train, how='left', left_index=True, right_index=True) \
                         .merge(vc_output_test, how='left', left_index=True, right_index=True).transpose()

# Print results
print("Max of (max - min) / (min): " + str(((vc_output.max() - vc_output.min()) / vc_output.min()).max()))
display(vc_output)

# Save results
dict_out['TVT split - Value counts output variable'] = vc_output.copy()

Max of (max - min) / (min): 0.024313737756419117


,1.0,2.0,3.0,4.0,5.0
overall_rating_x,0.081939,0.096617,0.203598,0.300255,0.317591
overall_rating_y,0.082096,0.096149,0.203660,0.300464,0.317630
overall_rating,0.081311,0.098487,0.203354,0.299415,0.317433


In [13]:
# Check all other variables
check_dataframes([reviews, reviews_train, reviews_test],
                 iqr_threshold=config['TVT split']['max allowed difference numerical iqr'],
                 cat_max=config['TVT split']['max allowed difference categorical'],
                 exclude_columns=config['TVT split']['collumns to skip'],
                 cats_ignore=config['TVT split']['category minimum percentage'])

All checks satisfactory.


## 4. Missing values

### 4.1. Missing value understanding
Only two types of collumns should have missing values:
   - Data gathered directly from Glassdoor: some of the questions are optional, so they might not all have responses
   - Linguistic dimension data: some reviews do not display all linguistic dimensions, so they are read in as missing values
    
Now, it will be checked whether this is actually what is happening

In [14]:
# Print non-ld columns which have missing values
print([col for col in reviews_train.isna().sum()[reviews_train.isna().sum() > 0].index.to_list() if "_ld_" not in col])

['summary', 'job_title', 'wl_balance', 'culture_values', 'diversity_inclusion', 'career_opportunities', 'compensation_benefits', 'senior_management', 'stock_percchange_month', 'stock_percchange_year']


### 4.2. Missing value treatment
The linguistic dimension columns will be filled with 0

In [15]:
# Save original column order
col_order = reviews_train.columns.to_list()

# Identify columns with "_ld_" in their names
ld_columns = [col for col in reviews_train.columns if "_ld_" in col]

# Define the transformer
transformer_mv = ColumnTransformer(
    transformers=[
        ('ld_imputer', SimpleImputer(strategy='constant', fill_value=config['missing values']['fill linguistic dimensions']), ld_columns)
    ],
    remainder='passthrough'  # Include columns not specified in transformers
)

# Fit transformer
transformer_mv.fit(reviews_train)

# Get the transformed column names
transformed_columns = transformer_mv.get_feature_names_out(input_features=reviews_train.columns)

# Apply the transformer to the original DataFrame
reviews_train = pd.DataFrame(transformer_mv.transform(reviews_train), columns=[col.split("__")[1] for col in transformed_columns])
reviews_train = reviews_train[col_order]

In [16]:
# Save object
dict_out['missing value transformer'] = transformer_mv

In [17]:
# Modify test dataset
transformed_columns = transformer_mv.get_feature_names_out(input_features=reviews_test.columns)

reviews_test = pd.DataFrame(transformer_mv.transform(reviews_test), columns=[col.split("__")[1] for col in transformed_columns])
reviews_test = reviews_test[col_order]

In [18]:
# Rest column types
reviews_train = reviews_train.infer_objects()
reviews_test = reviews_test.infer_objects()

In [19]:
# Print non-ld columns which have missing values
print("Non-ld columns with missing values:")
print([col for col in reviews_train.isna().sum()[reviews_train.isna().sum() > 0].index.to_list() if "_ld_" not in col])

# Print ld columns which have missing values
print("\nLd columns with missing values:")
print([col for col in reviews_train.isna().sum()[reviews_train.isna().sum() > 0].index.to_list() if "_ld_" in col])

Non-ld columns with missing values:
['summary', 'job_title', 'wl_balance', 'culture_values', 'diversity_inclusion', 'career_opportunities', 'compensation_benefits', 'senior_management', 'stock_percchange_month', 'stock_percchange_year']

Ld columns with missing values:
[]


## 5. Outliers

### 5.1. Outlier detection
Here, the outliers will be flagged. Outliers are defined as values which are, in absolute value, larger than x times the median (x is defined by the user).

In [20]:
def outlier_detection(df_train, multiplier, df_test):
    
    # Iterate through columns
    for col in df_train.columns:
        
        if not pd.api.types.is_numeric_dtype(df_train[col]):
            continue

        # Calculate outlier limit
        iqr = pd.Series(df_train[col].unique()).quantile(0.75) - (df_train[df_train[col] != 0][col]).quantile(0.25)
        lim_max = pd.Series(df_train[col]).quantile(0.75) + iqr * multiplier
        lim_min = pd.Series(df_train[col]).quantile(0.25) - iqr * multiplier
        
        # Calculate outliers
        outliers_train = df_train[(df_train[col] > lim_max) | (df_train[col] < lim_min)]
        outliers_test = df_test[(df_test[col] > lim_max) | (df_test[col] < lim_min)]

        # Print outliers
        if len(outliers_train) > 0:
            print(col + " has " + str(len(outliers_train) / len(df_train) * 100) + "% outliers in train")

        if len(outliers_test) > 0:
            print(col + " has " + str(len(outliers_test) / len(df_test) * 100) + "% outliers in test")

In [21]:
# Execute function
outlier_detection(reviews_train, config['outliers']['interquartile range multiplier'], reviews_test)

summary_charlen has 0.001402150899479802% outliers in test
cons_charlen has 0.00035053895364122335% outliers in train
summary_ld_PDT has 0.0007010779072824467% outliers in train
summary_ld_PDT has 0.001402150899479802% outliers in test
summary_ld_`` has 0.0007010779072824467% outliers in train
summary_ld_JJS has 0.1770221715888178% outliers in train
summary_ld_JJS has 0.18508391873133387% outliers in test
summary_ld_RBS has 100.0% outliers in train
summary_ld_RBS has 100.0% outliers in test
summary_ld_VBD has 0.0028043116291297868% outliers in train
summary_ld_VBD has 0.002804301798959604% outliers in test
summary_ld_UH has 0.08132503724476382% outliers in train
summary_ld_UH has 0.07852045037086891% outliers in test
summary_ld_VBN has 0.17772324949610027% outliers in train
summary_ld_VBN has 0.18227961693237427% outliers in test
summary_ld_RP has 0.05223030409254229% outliers in train
summary_ld_RP has 0.04907528148179307% outliers in test
summary_ld_JJR has 0.001752694768206117% outl

cons_ld_VBN has 1.7842432740338272% outliers in train
cons_ld_VBN has 1.742873568053394% outliers in test
cons_ld_UH has 0.027342038384015423% outliers in train
cons_ld_UH has 0.022434414391676833% outliers in test
cons_ld_RP has 3.262466041538866% outliers in train
cons_ld_RP has 3.3371191407619287% outliers in test
cons_ld_JJR has 7.376391201472264% outliers in train
cons_ld_JJR has 7.411769654650234% outliers in test
cons_ld_WDT has 1.997721496801332% outliers in train
cons_ld_WDT has 2.031716653346233% outliers in test
cons_ld_FW has 0.1318026465691% outliers in train
cons_ld_FW has 0.12479143005370238% outliers in test
cons_ld_, has 0.9829112260099905% outliers in train
cons_ld_, has 0.928223895455629% outliers in test
cons_ld_VBZ has 1.6219437384979405% outliers in train
cons_ld_VBZ has 1.5409638385283024% outliers in test
cons_ld_MD has 11.240732626413111% outliers in train
cons_ld_MD has 11.53970190271877% outliers in test
cons_ld_PRP has 0.00210323372184734% outliers in train


cons_empath_wealthy has 100.0% outliers in train
cons_empath_wealthy has 100.0% outliers in test
cons_empath_exercise has 3.2158443607045832% outliers in train
cons_empath_exercise has 3.1099706950462007% outliers in test
cons_empath_home has 4.376829375164315% outliers in train
cons_empath_home has 4.355080693784265% outliers in test
cons_empath_rage has 0.10936815353606169% outliers in train
cons_empath_rage has 0.10796561925994475% outliers in test
cons_empath_fun has 0.4672684252037508% outliers in train
cons_empath_fun has 0.4683184004262539% outliers in test
cons_empath_negative_emotion has 5.841030584523705% outliers in train
cons_empath_negative_emotion has 5.908663890407886% outliers in test
cons_empath_payment has 9.618087810007887% outliers in train
cons_empath_payment has 9.819262749057053% outliers in test
cons_empath_achievement has 9.204101305757602% outliers in train
cons_empath_achievement has 9.223348616778138% outliers in test
GDP per capita has 0.260450442555429% ou

### 5.2. Outlier treatment
No outlier treatment was done.

Other options: clipping, deletion

## 6. Feature engineering

### 6.1. Date and foundation
Now, three different variables will be constructed:
 - Days from review date to t0 date
 - Days from company foundation to t0 date
 - Days from company foundation to review date
 
The t0 date is a date defined by the user

In [22]:
def calculate_date_diffs(df, timestamp0):
    
    # Calculate days from review
    df['days from review to t0'] = (timestamp0 - pd.to_datetime(df['date'])).dt.days
    
    # Calculate days from foundation
    df['days from foundation to t0'] = (timestamp0 - pd.to_datetime(df['Founded'].astype(str) + "-01-01", format='%Y-%m-%d')).dt.days
    
    # Calculate dayf from foundation to review
    df['days from foundation to review'] = (pd.to_datetime(df['date']) - pd.to_datetime(df['Founded'].astype(str) + "-01-01", format='%Y-%m-%d')).dt.days
    df['days from foundation to review'] = np.where(df['days from foundation to review'] < 0, 0, df['days from foundation to review'])
    
    return df

In [23]:
# Obtain t0 and print
t0 = pd.to_datetime(config['feature engineering']['date to take as t0'])
print(t0)

2024-01-01 00:00:00


In [24]:
# Apply function to dataframes
reviews_train = calculate_date_diffs(reviews_train, t0)
reviews_test = calculate_date_diffs(reviews_test, t0)

## 7. Categorical variable treatment
### 7.1. Target encoding
The statename and sector variables will be encoded.

In [25]:
# Define target encoding function
def target_encoding(df_train, df_test, output_feature, column_to_encode, agg_functions):

    # Calculate group by
    df_train_targets = df_train[[column_to_encode, output_feature]].groupby(column_to_encode).agg(agg_functions).reset_index()

    # Fix multiindex
    df_train_targets.columns = [col[0] if col[1] == '' else '_'.join(col) for col in df_train_targets.columns]

    # Modify column names
    df_train_targets = df_train_targets.rename(columns={'overall_rating_' + agg_func: column_to_encode + "_targenc_" + agg_func for agg_func in agg_functions})

    # Get overall of aggregation functions
    dict_train_targetsoverall = {agg_func: getattr(df_train[output_feature], agg_func)() for agg_func in agg_functions}

    # Merge with different dataframes
    df_train = pd.merge(df_train, df_train_targets, on=column_to_encode, how='left')
    df_test = pd.merge(df_test, df_train_targets, on=column_to_encode, how='left')

    # Fill missing values
    cols_to_check = [column_to_encode + "_targenc_" + agg_func for agg_func in agg_functions]

    for col in cols_to_check:

        # Print number of values to fill in in total
        print("Number of missing values before filling in " + col + ": " + str(df_train[col].isna().sum() + df_test[col].isna().sum()))

        # Fill in missing values
        df_train[col] = df_train[col].fillna(dict_train_targetsoverall[col.split("_")[-1]])
        df_test[col] = df_test[col].fillna(dict_train_targetsoverall[col.split("_")[-1]])

        # Print number of missing values after fillin
        print("Number of missing values after filling in " + col + ": " + str(df_train[col].isna().sum() + df_test[col].isna().sum()) + "\n")
        
    # Return
    return [df_train, df_test, df_train_targets, dict_train_targetsoverall]

In [26]:
## Apply target encoding function

# Initialize dictionary for output
dict_out['categorical features'] = {'target encoding': {}}

# Iterate through features to encode
for feature in config['categorical features']['target encoding']['features']:
    
    # Apply encoding function
    lst_targenc = target_encoding(reviews_train, reviews_test, config['categorical features']['target encoding']['output feature'], feature, config['categorical features']['target encoding']['operations'])
    
    reviews_train = lst_targenc[0].copy()
    reviews_test = lst_targenc[1].copy()
    dict_out['categorical features']['target encoding'][feature] = {'group by feature': lst_targenc[2].copy(), 'overall':lst_targenc[3].copy()}

Number of missing values before filling in State name_targenc_median: 0
Number of missing values after filling in State name_targenc_median: 0

Number of missing values before filling in State name_targenc_mean: 0
Number of missing values after filling in State name_targenc_mean: 0

Number of missing values before filling in Sector_targenc_median: 0
Number of missing values after filling in Sector_targenc_median: 0

Number of missing values before filling in Sector_targenc_mean: 0
Number of missing values after filling in Sector_targenc_mean: 0



### 7.2. One hot encoding

In [27]:
# Define encoder
cols_to_ohe = config['categorical features']['one hot encoding']['features']
ohe_encoder = OneHotEncoder(sparse=False, handle_unknown=config['categorical features']['one hot encoding']['settings']['handle_unkown'])
ohe_encoder.fit(reviews_train[cols_to_ohe])

# Transform columns
encoded_values_train = ohe_encoder.transform(reviews_train[cols_to_ohe])
reviews_train_ohe = pd.DataFrame(encoded_values_train, columns=[feature + "_ohe" for feature in ohe_encoder.get_feature_names_out()])

encoded_values_test = ohe_encoder.transform(reviews_test[cols_to_ohe])
reviews_test_ohe = pd.DataFrame(encoded_values_test, columns=[feature + "_ohe" for feature in ohe_encoder.get_feature_names_out()])

# Concatenate the original DataFrame and the one-hot encoded DataFrame
reviews_train = pd.concat([reviews_train, reviews_train_ohe], axis=1)
reviews_test = pd.concat([reviews_test, reviews_test_ohe], axis=1)

# Save one hot encoder
dict_out['one hot encoder'] = ohe_encoder

## 8. Scaling

In [28]:
# Define separate function
def separate_numerical_categorical(df):
    
    # Set overall rating as categorical variable
    df['overall_rating'] = df['overall_rating'].astype('category')
    
    # Select numerical variables
    numerical_df = df.select_dtypes(include=['number'])

    # Select non-numerical variables
    categorical_df = df.select_dtypes(exclude=['number'])

    return numerical_df, categorical_df

In [29]:
def check_same_columns_in_order(df1, df2):
    columns_df1 = list(df1.columns)
    columns_df2 = list(df2.columns)

    if columns_df1 == columns_df2:
        return True
    else:
        return False

In [30]:
def concatenate_dataframes(df1, df2):
    # Check if dataframes have the same length
    if len(df1) != len(df2):
        raise ValueError("Dataframes are not the same length and cannot concatenate")

    # Concatenate horizontally if the dataframes have the same length
    result_df = pd.concat([df1, df2], axis=1)

    return result_df

In [31]:
# Separate categorical and numerical variables
[reviews_train_num, reviews_train_cat] = separate_numerical_categorical(reviews_train)
[reviews_test_num, reviews_test_cat] = separate_numerical_categorical(reviews_test)

# Check that all the columns are the same
num_same = check_same_columns_in_order(reviews_train_num, reviews_test_num)
cat_same = check_same_columns_in_order(reviews_train_cat, reviews_test_cat)

if not(num_same & cat_same):
    raise ValueError("Values are not the same")

In [32]:
# Train scaler and transform
scaler_st = StandardScaler()
reviews_train_num_scaled = pd.DataFrame(scaler_st.fit_transform(reviews_train_num), columns=scaler_st.get_feature_names_out())

reviews_test_num_scaled = pd.DataFrame(scaler_st.transform(reviews_test_num), columns=scaler_st.get_feature_names_out())

# Save scaler
dict_out['standard scaler'] = scaler_st

In [33]:
# Concatenate categorical and numerical columns
reviews_train_scaled = concatenate_dataframes(reviews_train_cat, reviews_train_num_scaled)
reviews_test_scaled = concatenate_dataframes(reviews_test_cat, reviews_test_num_scaled)

## 9. Export outputs
Three different types of outputs will be exported:
  - A pickle object with:
      - The train - test split value counts of output variable
      - The missing value transformer
      - The categorical features transformer
      - The one hot encoder
      - The standard scaler
      
  - An Excel file with each of the three dataframes
  - A json document analogous to the config file  

In [34]:
# Get current datetime
current_datetime = datetime.now().strftime("%Y%m%d_%H%M%S")

# Create empty directory
directory_name = "03_outputs_data_preprocessing/output_data_preprocessing_" + current_datetime
os.mkdir(directory_name)

In [35]:
# Create pickle object
pickle_file_path = directory_name + "/dict_" + current_datetime + ".pkl"

with open(pickle_file_path, 'wb') as pickle_file:
    pickle.dump(dict_out, pickle_file)

In [36]:
# Create filepath
csv_file_path = directory_name + "/dataframes_" + current_datetime + ".xlsx"

# Save the different sheets
sheet_name = "Train"
csv_file_path = directory_name + "/dataframes_" + sheet_name + "_" + current_datetime + ".csv"
reviews_train_scaled.to_csv(csv_file_path, index=False)

sheet_name = "Test"
csv_file_path = directory_name + "/dataframes_" + sheet_name + "_" + current_datetime + ".csv"
reviews_test_scaled.to_csv(csv_file_path, index=False)

In [37]:
json_file_path = directory_name + "/config_" + current_datetime + ".json"

# Export the dictionary to a JSON file
with open(json_file_path, 'w') as json_file:
    json.dump(config, json_file)

In [38]:
# Print
print("Everything worked!")

Everything worked!
